In [1]:
# ==========================================
# CELL 1: Extract Text Features via Whisper
# ==========================================
import os
import numpy as np
import whisper
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

PROJECT_PATH = "Thesis_Data"
VIDEO_DIR = f"{PROJECT_PATH}/videos"

print("Loading Whisper and Text Embedding models...")
# 'base' is fast and accurate enough for sentiment. 
# You can change to 'small' or 'medium' if you have a strong GPU.
whisper_model = whisper.load_model("base") 
text_model = SentenceTransformer('all-MiniLM-L6-v2')

text_features_dict = {}
video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith('.mp4')]

print(f"Processing {len(video_files)} videos for text...")
for v_file in tqdm(video_files):
    v_id = v_file.replace(".mp4", "")
    v_path = os.path.join(VIDEO_DIR, v_file)
    
    try:
        # 1. Transcribe the audio to text
        result = whisper_model.transcribe(v_path, fp16=False)
        transcript = result['text'].strip()
        
        # 2. Convert text to embeddings (384 dimensions)
        # If transcript is empty, it still creates a valid mathematical vector
        embedding = text_model.encode(transcript)
        
        text_features_dict[v_id] = embedding
        
    except Exception as e:
        print(f"Error on {v_id}: {e}")
        # Fallback to zeros if extraction fails completely
        text_features_dict[v_id] = np.zeros(384)

# 3. Save the new features
np.save(f"{PROJECT_PATH}/text_features_whisper.npy", text_features_dict)
print("Text features saved successfully!")

Loading Whisper and Text Embedding models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing 486 videos for text...


  0%|          | 0/486 [00:00<?, ?it/s]

Text features saved successfully!


In [2]:
# ==========================================
# CELL 2: Triple-Branch PCA & XGBoost (3-Class)
# ==========================================
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

# ------------------------------------------
# 1. LOAD ALL 3 MODALITIES
# ------------------------------------------
PROJECT_PATH = "Thesis_Data"
df = pd.read_csv("videos_with_sentiment_labels.csv")

visual_dict = np.load(f"{PROJECT_PATH}/visual_features_clip.npy", allow_pickle=True).item()
audio_dict = np.load(f"{PROJECT_PATH}/audio_features_vggish.npy", allow_pickle=True).item()
text_dict = np.load(f"{PROJECT_PATH}/text_features_whisper.npy", allow_pickle=True).item()

X_v_list, X_a_list, X_t_list, y_labels = [], [], [], []

for index, row in df.iterrows():
    v_id = row['video_id']
    label = row['majority_sentiment']
    
    # Only keep the video if we successfully extracted all 3 modalities
    if v_id in visual_dict and v_id in audio_dict and v_id in text_dict:
        X_v_list.append(visual_dict[v_id])
        X_a_list.append(audio_dict[v_id])
        X_t_list.append(text_dict[v_id])
        y_labels.append(label)

X_visual = np.array(X_v_list)
X_audio = np.array(X_a_list)
X_text = np.array(X_t_list)
y_labels = np.array(y_labels)

# Max Pooling (for any 3D arrays like the original CLIP/VGGish)
def apply_max_pooling(features):
    if features.ndim == 3: return np.max(features, axis=1) 
    return features

X_visual = apply_max_pooling(X_visual)
X_audio = apply_max_pooling(X_audio)
# Text embeddings are already 1D (per video) from SentenceTransformers

le = LabelEncoder()
y_encoded = le.fit_transform(y_labels)

print(f"Total videos fused: {len(y_encoded)}")
print("Beginning Triple-Branch PCA Training...\n")

# ------------------------------------------
# 2. TRIPLE-BRANCH PCA PIPELINE
# ------------------------------------------
k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

fold_accs = []
all_true, all_preds = [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_visual, y_encoded)):
    
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    # --- BRANCH 1: VISUAL (PCA down to 10) ---
    scaler_v = StandardScaler()
    v_train = scaler_v.fit_transform(X_visual[train_idx])
    v_val = scaler_v.transform(X_visual[val_idx])
    pca_v = PCA(n_components=10, random_state=42)
    v_train_pca = pca_v.fit_transform(v_train)
    v_val_pca = pca_v.transform(v_val)

    # --- BRANCH 2: AUDIO (PCA down to 5) ---
    scaler_a = StandardScaler()
    a_train = scaler_a.fit_transform(X_audio[train_idx])
    a_val = scaler_a.transform(X_audio[val_idx])
    pca_a = PCA(n_components=5, random_state=42)
    a_train_pca = pca_a.fit_transform(a_train)
    a_val_pca = pca_a.transform(a_val)

    # --- BRANCH 3: TEXT (PCA down to 15) ---
    scaler_t = StandardScaler()
    t_train = scaler_t.fit_transform(X_text[train_idx])
    t_val = scaler_t.transform(X_text[val_idx])
    pca_t = PCA(n_components=15, random_state=42)
    t_train_pca = pca_t.fit_transform(t_train)
    t_val_pca = pca_t.transform(t_val)
    
    # --- FINAL FUSION ---
    # Glue the branches together (10 + 5 + 15 = 30 columns)
    X_train_fused = np.concatenate((v_train_pca, a_train_pca, t_train_pca), axis=1)
    X_val_fused = np.concatenate((v_val_pca, a_val_pca, t_val_pca), axis=1)
    
    # --- TRAIN XGBOOST ---
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    
    xgb_model = XGBClassifier(
        n_estimators=200, 
        objective='multi:softprob', 
        num_class=len(le.classes_),
        eval_metric='mlogloss',
        random_state=42, 
        n_jobs=-1
    )
    
    xgb_model.fit(X_train_fused, y_train, sample_weight=sample_weights)
    fold_preds = xgb_model.predict(X_val_fused)
    
    fold_accs.append(accuracy_score(y_val, fold_preds))
    all_true.extend(y_val)
    all_preds.extend(fold_preds)

# ------------------------------------------
# 3. RESULTS
# ------------------------------------------
print("="*50)
print("=== TEXT + VISUAL + AUDIO FUSION RESULTS ===")
print("="*50)
print(f"Average Accuracy: {np.mean(fold_accs):.4f} (+/- {np.std(fold_accs):.4f})\n")
print(classification_report(all_true, all_preds, target_names=le.classes_))

Total videos fused: 446
Beginning Triple-Branch PCA Training...

=== TEXT + VISUAL + AUDIO FUSION RESULTS ===
Average Accuracy: 0.5135 (+/- 0.0317)

              precision    recall  f1-score   support

    Negative       0.16      0.06      0.08        52
     Neutral       0.36      0.32      0.34       136
    Positive       0.60      0.71      0.65       258

    accuracy                           0.51       446
   macro avg       0.37      0.36      0.36       446
weighted avg       0.47      0.51      0.49       446



In [7]:
# ==========================================
# MASTER CELL: PyTorch Multimodal Fusion (Text + Vis + Audio)
# ==========================================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------
# 1. LOAD DATA
# ------------------------------------------
PROJECT_PATH = "Thesis_Data"
df = pd.read_csv("videos_with_sentiment_labels.csv")

visual_dict = np.load(f"{PROJECT_PATH}/visual_features_clip.npy", allow_pickle=True).item()
audio_dict = np.load(f"{PROJECT_PATH}/audio_features_vggish.npy", allow_pickle=True).item()
text_dict = np.load(f"{PROJECT_PATH}/text_features_whisper.npy", allow_pickle=True).item()

X_v_list, X_a_list, X_t_list, y_labels = [], [], [], []

for index, row in df.iterrows():
    v_id = row['video_id']
    label = row['majority_sentiment']
    
    if v_id in visual_dict and v_id in audio_dict and v_id in text_dict:
        X_v_list.append(visual_dict[v_id])
        X_a_list.append(audio_dict[v_id])
        X_t_list.append(text_dict[v_id])
        y_labels.append(label)

def apply_max_pooling(features):
    if features.ndim == 3: return np.max(features, axis=1) 
    return features

X_visual = apply_max_pooling(np.array(X_v_list)) # 512 dims
X_audio = apply_max_pooling(np.array(X_a_list))   # 128 dims
X_text = np.array(X_t_list)                       # 384 dims
y_labels = np.array(y_labels)

le = LabelEncoder()
y_encoded = le.fit_transform(y_labels)

# ------------------------------------------
# 2. THE MULTIMODAL NEURAL NETWORK
# ------------------------------------------
class MultimodalFusionNet(nn.Module):
    def __init__(self, num_classes=3):
        super(MultimodalFusionNet, self).__init__()
        
        # 1. Branch Encoders (Compressing each modality intelligently)
        # Extreme dropout prevents the 446 videos from being memorized
        self.vis_encoder = nn.Sequential(
            nn.Linear(512, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.6)
        )
        self.aud_encoder = nn.Sequential(
            nn.Linear(128, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.5)
        )
        self.txt_encoder = nn.Sequential(
            nn.Linear(384, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.6) # Text gets largest capacity
        )
        
        # 2. Fusion Classifier (64 + 32 + 128 = 224 combined features)
        self.classifier = nn.Sequential(
            nn.Linear(224, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, v, a, t):
        v_feat = self.vis_encoder(v)
        a_feat = self.aud_encoder(a)
        t_feat = self.txt_encoder(t)
        
        # Glue them together!
        fused = torch.cat((v_feat, a_feat, t_feat), dim=1)
        return self.classifier(fused)

# ------------------------------------------
# 3. TRAINING LOOP WITH CLASS WEIGHTS
# ------------------------------------------
k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

# FORCE the network to pay attention to the 52 Negative videos
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

fold_accs = []
all_true, all_preds = [], []
MAX_EPOCHS = 150
PATIENCE = 20

print("Starting Deep Multimodal Fusion (5-Fold CV)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_visual, y_encoded)):
    print(f"\n--- FOLD {fold + 1} ---")
    
    # Scale each modality separately
    scaler_v, scaler_a, scaler_t = StandardScaler(), StandardScaler(), StandardScaler()
    
    v_train = scaler_v.fit_transform(X_visual[train_idx])
    v_val = scaler_v.transform(X_visual[val_idx])
    
    a_train = scaler_a.fit_transform(X_audio[train_idx])
    a_val = scaler_a.transform(X_audio[val_idx])
    
    t_train = scaler_t.fit_transform(X_text[train_idx])
    t_val = scaler_t.transform(X_text[val_idx])
    
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    # Create DataLoaders handling 3 inputs
    train_loader = DataLoader(TensorDataset(
        torch.tensor(v_train, dtype=torch.float32), 
        torch.tensor(a_train, dtype=torch.float32), 
        torch.tensor(t_train, dtype=torch.float32), 
        torch.tensor(y_train, dtype=torch.long)
    ), batch_size=32, shuffle=True)
    
    val_loader = DataLoader(TensorDataset(
        torch.tensor(v_val, dtype=torch.float32), 
        torch.tensor(a_val, dtype=torch.float32), 
        torch.tensor(t_val, dtype=torch.float32), 
        torch.tensor(y_val, dtype=torch.long)
    ), batch_size=32, shuffle=False)
    
    model = MultimodalFusionNet(num_classes=len(le.classes_))
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3) # L2 Regularization
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(MAX_EPOCHS):
        model.train()
        for v, a, t, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(v, a, t)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for v, a, t, labels in val_loader:
                outputs = model(v, a, t)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_state = model.state_dict()
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= PATIENCE:
            break
            
    # Evaluate best epoch
    model.load_state_dict(best_model_state)
    model.eval()
    
    with torch.no_grad():
        for v, a, t, labels in val_loader:
            outputs = model(v, a, t)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_true.extend(labels.numpy())

# ------------------------------------------
# 4. FINAL VERDICT
# ------------------------------------------
print("\n" + "="*50)
print("=== DEEP MULTIMODAL NN RESULTS ===")
print("="*50)
final_acc = accuracy_score(all_true, all_preds)
print(f"Overall Accuracy: {final_acc:.4f}\n")
print(classification_report(all_true, all_preds, target_names=le.classes_))

Starting Deep Multimodal Fusion (5-Fold CV)...

--- FOLD 1 ---

--- FOLD 2 ---

--- FOLD 3 ---

--- FOLD 4 ---

--- FOLD 5 ---

=== DEEP MULTIMODAL NN RESULTS ===
Overall Accuracy: 0.5135

              precision    recall  f1-score   support

    Negative       0.17      0.23      0.19        52
     Neutral       0.42      0.48      0.45       136
    Positive       0.69      0.59      0.64       258

    accuracy                           0.51       446
   macro avg       0.43      0.43      0.43       446
weighted avg       0.55      0.51      0.53       446

